In [1]:
import requests
from collections import Counter


class APIError(Exception):
    pass

class CatFactProcessor:
    def __init__(self):
        self.last_fact = ""
    def get_fact(self):
        try:
            response = requests.get("https://catfact.ninja/fact")
            response.raise_for_status()
            data = response.json()
            self.last_fact = data["fact"]
            return self.last_fact
        except requests.exceptions.RequestException as e:
            raise APIError(f"Ошибка при запросе к API: {e}") from e
    def get_fact_analysis(self):
        if not self.last_fact:
            return {"length": 0, "letter_frequencies": {}}
        fact_length = len(self.last_fact)
        letter_frequencies = dict(Counter(self.last_fact.lower()))
        
        return {
            "length": fact_length,
            "letter_frequencies": letter_frequencies,
 }

In [3]:
import unittest
from unittest.mock import patch, Mock


class TestCatFactProcessor(unittest.TestCase):
    """Тесты для класса CatFactProcessor."""

    @patch('requests.get')
    def test_get_fact_success(self, mock_get):
        """
        Позитивный тест: проверка, что get_fact возвращает
        корректный факт и сохраняет его в last_fact.
        """
        # Мокаем успешный ответ API
        mock_response = Mock()
        mock_response.raise_for_status.return_value = None
        mock_response.json.return_value = {"fact": "Cats purr loudly."}
        mock_get.return_value = mock_response

        processor = CatFactProcessor()
        fact = processor.get_fact()

        self.assertEqual(fact, "Cats purr loudly.")
        self.assertEqual(processor.last_fact, "Cats purr loudly.")

    @patch('requests.get')
    def test_get_fact_failure(self, mock_get):
        """
        Негативный тест: проверка, что при ошибке запроса
        выбрасывается пользовательское исключение APIError.
        """
        # Мокаем ошибку при запросе
        mock_get.side_effect = requests.exceptions.RequestException("Connection error")

        processor = CatFactProcessor()

        with self.assertRaises(APIError) as context:
            processor.get_fact()

        self.assertIn("Ошибка при запросе к API", str(context.exception))

    def test_fact_analysis_empty(self):
        """
        Тест: если факт не получен, анализ возвращает длину 0
        и пустой словарь частот.
        """
        processor = CatFactProcessor()
        result = processor.get_fact_analysis()

        expected = {"length": 0, "letter_frequencies": {}}
        self.assertEqual(result, expected)

    def test_fact_analysis_valid(self):
        """
        Позитивный тест: проверка корректного анализа
        строки last_fact на длину и частоты букв.
        """
        processor = CatFactProcessor()
        processor.last_fact = "Cat"

        result = processor.get_fact_analysis()

        expected_frequencies = {"c": 1, "a": 1, "t": 1}
        self.assertEqual(result["length"], 3)
        self.assertEqual(result["letter_frequencies"], expected_frequencies)


# Запуск тестов в Jupyter Notebook
unittest.main(argv=[''], exit=False)


....
----------------------------------------------------------------------
Ran 4 tests in 0.008s

OK
